# Mean cross-sections and maps for a group of simulations

Time-mean zonal-mean and equatorial cross-sections for several experiments
from a group in `common.GROUPS`, one panel per experiment with a shared colour scale,
following the plots in `Compare-Simulations.ipynb`.

As in `quicklook.py`, the vertical axis of each panel is the time-mean air pressure
interpolated to the levels of the diagnostic and reduced in the same way
(zonal mean or equatorial slice), so it varies along the horizontal axis.

Alternatively, with `VERT_COORD = "sigma"` the fields are interpolated to surfaces of
$\sigma_p=p/p_s$ before being reduced, as in `interp_to_sigma_p.py`.

The time-mean longitude-latitude maps are on a pressure level, with horizontal wind streamlines,
following `Compare-Simulations.ipynb`.

In [1]:
import math
import warnings

import iris
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import paths
from aeolus.calc import time_mean, zonal_mean
from aeolus.coord import roll_cube_pm180
from aeolus.model import lfric
from aeolus.plot import figsave, tex2cf_units, unit_format, subplot_label_generator
from common import (
    EXPERIMENTS,
    GROUPS,
    PA_TO_BAR,
    SIGMA_LEVELS,
    Diag,
    common_kw_plt,
    interp_to_levels_of,
    interp_to_plev,
    interp_to_sigma_levels,
    load_exp_days,
    setup_latitude_axis,
    setup_longitude_axis,
    setup_pressure_axis,
)

warnings.filterwarnings("ignore", category=UserWarning, module="iris")
iris.FUTURE.date_microseconds = True

## Experiments

Experiments are stored in `data/lfric/<group>/<label>/`.

In [2]:
for group_key, group in GROUPS.items():
    print(f"{group_key}: {group.title}")
    print(*(f"    {exp_key}" for exp_key in group.simulations), sep="\n")

shj: Shallow Hot Jupiter
    shj_c48_l32
    shj_c24_l32
    shj_c96_l32
    shj_c48_l16
    shj_c48_l64
dhj: Deep Hot Jupiter
    dhj_c48_l66
    dhj_c24_l66
    dhj_c96_l66
dhj_s: Deep Hot Jupiter, Stretched Mesh
    dhj_c48_l66_s0p5_m90
    dhj_c48_l66_s0p5_p90
dhj_all: Deep Hot Jupiter
    dhj_c48_l66
    dhj_c24_l66
    dhj_c96_l66
    dhj_c48_l66_s0p5_m90
    dhj_c48_l66_s0p5_p90
shj_dhj_all: Shallow & Deep Hot Jupiter
    shj_c48_l32
    shj_c24_l32
    shj_c96_l32
    shj_c48_l16
    shj_c48_l64
    dhj_c48_l66
    dhj_c24_l66
    dhj_c96_l66
    dhj_c48_l66_s0p5_m90
    dhj_c48_l66_s0p5_p90
hd209_all: HD 209458b (all)
    hd209_dt30_nrad3
    hd209_dt30_nrad4
    hd209_dt30_nrad5
    hd209_rand_t100k_nf20_tau1e5_amp1e-2_sclht6
hd209: HD 209458b
    hd209_dt30_nrad3
    hd209_dt30_nrad4
    hd209_dt30_nrad5


In [ ]:
GROUP_KEY = "shj"
# Subset of the group's simulations to plot (all of them if None)
CASES = None
N_DAYS_MEAN = 200  # length of the averaging period
NCOLS = 5  # number of panels per row
LINEAR_PRESSURE = True  # linear instead of log pressure axis
VERT_COORD = "sigma"  # vertical coordinate: "pressure" or "sigma" (p/p_s)
N_CONTOURS = (
    7  # approximate number of contour lines on top of the shading; 0 to disable
)
SAVE_FIGS = True

# The default levels stop at 0.01, well below the top of the deep atmospheres, where
# the pressure is not monotonic with height in some columns above sigma ~ 3e-6
SIGMA_LEVS = SIGMA_LEVELS if GROUP_KEY == "shj" else np.geomspace(1, 1e-5, 67)

EXPS = list(GROUPS[GROUP_KEY].simulations) if CASES is None else CASES
# Average all experiments over the same days, at the end of the shortest run
END_DAY = min(EXPERIMENTS[exp_key].run_length for exp_key in EXPS)
START_DAY = END_DAY - N_DAYS_MEAN
PERIOD_STR = f"days {START_DAY}-{END_DAY}"
print(*EXPS, sep="\n")
print(PERIOD_STR)

In [ ]:
ds_all = {}
for exp_key in EXPS:
    ds_all[exp_key] = time_mean(load_exp_days(exp_key, START_DAY, END_DAY))
    print(exp_key)

In [ ]:
# White outline of the black contour lines and labels, to stand out against dark colours
KW_PATH_EFFECTS = [mpe.withStroke(linewidth=1, foreground="w")]


def exp_title(exp_key):
    """Experiment title, or its label if the titles in `EXPS` are not unique."""
    titles = [EXPERIMENTS[k].title for k in EXPS]
    if len(set(titles)) < len(titles):
        return EXPERIMENTS[exp_key].label
    return EXPERIMENTS[exp_key].title


def make_axes(fig, n_panels, ncols=NCOLS):
    """Grid of panels sharing the y-axis; unused panels are removed."""
    ncols = min(ncols, n_panels)
    nrows = math.ceil(n_panels / ncols)
    axs = fig.subplots(nrows, ncols, sharey=True, squeeze=False).ravel()
    for ax in axs[n_panels:]:
        ax.remove()
    return axs[:n_panels]


def add_contours(ax, x, y, data, levels):
    """Draw labelled black contour lines with a white outline."""
    cs = ax.contour(
        x,
        y,
        data,
        levels,
        colors="k",
        linewidths=0.5,
        path_effects=KW_PATH_EFFECTS,
    )
    for label in ax.clabel(cs, fmt="%g", fontsize="xx-small"):
        label.set_path_effects(KW_PATH_EFFECTS)


def pres_bar_like(cube, exp_key, collapse_func):
    """Time-mean pressure (bar) on the vertical grid of `cube`, reduced by `collapse_func`.

    Used as the 2D vertical coordinate of a cross-section, as in `quicklook.py`.
    """
    pres = ds_all[exp_key].extract_cube(lfric.pres)
    pres = collapse_func(interp_to_levels_of(pres, cube))
    pres.convert_units("Pa")
    return pres.data * PA_TO_BAR


def cross_section(cube, exp_key, collapse_func, vert_coord):
    """Reduce `cube` to a cross-section and get its 2D vertical coordinate."""
    if vert_coord == "sigma":
        pres = ds_all[exp_key].extract_cube(lfric.pres)
        xsec = collapse_func(interp_to_sigma_levels(cube, pres, SIGMA_LEVS))
        return xsec, np.broadcast_to(SIGMA_LEVS[:, np.newaxis], xsec.shape)
    if vert_coord == "pressure":
        return collapse_func(cube), pres_bar_like(cube, exp_key, collapse_func)
    raise ValueError(f"Unknown vertical coordinate: {vert_coord}")


def plot_cross_sections(
    diag_key,
    collapse_func,
    x_coord,
    suptitle,
    fname_suffix,
    ncol=NCOLS,
    add_cbar=True,
    linear_pressure=LINEAR_PRESSURE,
    vert_coord=VERT_COORD,
):
    """Plot a `vert_coord`-`x_coord` cross-section for each experiment in `EXPS`.

    `collapse_func` reduces a 3D (z, y, x) cube to the 2D cross-section.
    """
    diag = DIAGS[diag_key]
    cubes, vert_vals = [], []
    for exp_key in EXPS:
        cube, vert = cross_section(
            diag.recipe(ds_all[exp_key]), exp_key, collapse_func, vert_coord
        )
        cube.convert_units(tex2cf_units(diag.units))
        cubes.append(cube)
        vert_vals.append(vert)
    kw_plt, levels = common_kw_plt(diag, cubes, n_contours=N_CONTOURS)
    # Common vertical range of all panels
    vert_lim = (min(v.min() for v in vert_vals), max(v.max() for v in vert_vals))
    vert_label = {} if vert_coord == "pressure" else {"label": r"$\sigma_p=p/p_s$"}

    ncols = min(ncol, len(EXPS))
    nrows = math.ceil(len(EXPS) / ncols)
    fig = plt.figure(figsize=(2 * ncols, 2 * nrows), layout="constrained")
    fig.suptitle(
        f"{suptitle} {diag.title} / {diag.units} ({PERIOD_STR})", fontsize="medium"
    )
    axs = make_axes(fig, len(EXPS), ncols=ncols)

    iletters = subplot_label_generator()
    for i, (ax, exp_key, cube, vert) in enumerate(zip(axs, EXPS, cubes, vert_vals)):
        ax.set_title(
            f"({next(iletters)}) {exp_title(exp_key)}",
            fontsize="medium",
            color=EXPERIMENTS[exp_key].kw_plt["color"],
        )
        x_2d = np.broadcast_to(cube.coord(x_coord).points, vert.shape)
        im = getattr(ax, diag.method)(x_2d, vert, cube.data, **kw_plt)
        if N_CONTOURS:
            add_contours(ax, x_2d, vert, cube.data, levels)
        if x_coord == lfric.y:
            setup_latitude_axis(ax, which="x")
            ax.set_xticks(np.arange(-90, 91, 30))
        else:
            setup_longitude_axis(ax)
            ax.set_xticks(np.arange(0, 361, 60))
        setup_pressure_axis(ax, vert_lim, linear=linear_pressure, **vert_label)
        if i % ncols:
            ax.set_ylabel("")
        ax.grid(True, linestyle="--", alpha=0.5, dash_capstyle="round")
    # Colour bar ticks at the contour levels
    if add_cbar:
        fig.colorbar(
            im,
            ax=axs,
            orientation="vertical",
            extend="both",
            ticks=levels,
            # shrink=0.5,
            # aspect=40,
        )

    if SAVE_FIGS:
        figsave(
            fig,
            paths.drafts / "xsec" / f"{GROUP_KEY}__{diag_key}_{fname_suffix}"
            f"{'_sigma' if vert_coord == 'sigma' else ''}"
            f"__days_{START_DAY}_{END_DAY}",
        )
    return fig

In [ ]:
DIAGS = {
    "temp": Diag(
        title="Air Temperature",
        units="$K$",
        recipe=lambda cl: cl.extract_cube("air_temperature"),
        kw_plt={"cmap": "plasma"},
    ),
    "u": Diag(
        title="Eastward Wind",
        units="$m$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("eastward_wind"),
        cnorm=True,
        kw_plt={"cmap": "RdBu_r", "vmin": -400, "vmax": 1000}
        if GROUP_KEY == "shj"
        else {"cmap": "RdBu_r", "vmin": -2000, "vmax": 10_000},
    ),
    "w": Diag(
        title="Upward Wind",
        units="$m$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("upward_air_velocity"),
        cnorm=True,
        kw_plt={"cmap": "PuOr_r"},
    ),
}

Fields are shaded with `Diag.method` (`pcolormesh` by default) with `N_CONTOURS` contour lines on top, as in `quicklook.py`.
Colour limits and contour levels are shared by all panels and computed from the data unless `vmin`/`vmax` or `levels` are set in `kw_plt` of a diagnostic.
If `cnorm=True`, colours are centred on zero: with both `vmin` and `vmax` set, `TwoSlopeNorm` is used (needs `vmin < 0 < vmax`); otherwise the limits are symmetric, e.g. `vmin = -vmax` if only `vmax` is set.

## Zonal-mean cross-sections

In [ ]:
for diag_key in ["u", "temp"]:
    plot_cross_sections(
        diag_key,
        collapse_func=zonal_mean,
        x_coord=lfric.y,
        suptitle="Zonal Mean",
        fname_suffix="zm",
        ncol=5,
        add_cbar=False,
    )

## Equatorial cross-sections

In [ ]:
for diag_key in ["u", "temp", "w"]:
    plot_cross_sections(
        diag_key,
        collapse_func=lambda cube: cube.interpolate(
            [(lfric.y, 0)], iris.analysis.Linear()
        ),
        x_coord=lfric.x,
        suptitle="Equatorial Cross-Section of",
        fname_suffix="eq",
        add_cbar=diag_key == "w",
    )

## Longitude-latitude maps

In [ ]:
LEVEL = 1e4  # Pa
OVERLAY_STREAM = True


def to_plev(cube, exp_key, level=LEVEL):
    """Interpolate a time-mean cube to a pressure level, centred on 0 deg longitude."""
    return roll_cube_pm180(
        interp_to_plev(cube, ds_all[exp_key].extract_cube(lfric.pres), level),
        model=lfric,
    )


def plot_maps(
    diag_key,
    level=LEVEL,
    overlay_stream=OVERLAY_STREAM,
    ncol=NCOLS,
    add_cbar=True,
):
    """Plot a longitude-latitude map on a pressure level for each experiment in `EXPS`."""
    diag = DIAGS[diag_key]
    cubes = []
    for exp_key in EXPS:
        cube = to_plev(diag.recipe(ds_all[exp_key]), exp_key, level=level)
        cube.convert_units(tex2cf_units(diag.units))
        cubes.append(cube)
    kw_plt, levels = common_kw_plt(diag, cubes, n_contours=N_CONTOURS)

    ncols = min(ncol, len(EXPS))
    nrows = math.ceil(len(EXPS) / ncols)
    fig = plt.figure(figsize=(2.5 * ncols, 1.5 * nrows + 0.5), layout="constrained")
    axs = make_axes(fig, len(EXPS), ncols=ncols)

    iletters = subplot_label_generator()
    for i, (ax, exp_key, cube) in enumerate(zip(axs, EXPS, cubes)):
        ax.set_title(
            f"({next(iletters)}) {exp_title(exp_key)}",
            fontsize="medium",
            color=EXPERIMENTS[exp_key].kw_plt["color"],
        )
        lons, lats = cube.coord(lfric.x).points, cube.coord(lfric.y).points
        im = getattr(ax, diag.method)(lons, lats, cube.data, **kw_plt)
        if N_CONTOURS:
            add_contours(ax, lons, lats, cube.data, levels)
        if overlay_stream:
            _u = to_plev(ds_all[exp_key].extract_cube("eastward_wind"), exp_key, level)
            _v = to_plev(ds_all[exp_key].extract_cube("northward_wind"), exp_key, level)
            wspd = np.hypot(_u.data, _v.data)
            ax.streamplot(
                *np.meshgrid(_u.coord(lfric.x).points, _u.coord(lfric.y).points),
                _u.data,
                _v.data,
                color="w",
                linewidth=wspd / wspd.max(),
                arrowstyle="wedge",
                arrowsize=0.75,
            )
        setup_latitude_axis(ax)
        ax.set_yticks(np.arange(-90, 91, 30))
        setup_longitude_axis(ax, pm180=True)
        ax.set_xticks(np.arange(-180, 181, 60))
        if i % ncols:
            ax.set_ylabel("")
        ax.grid(True, linestyle="--", alpha=0.5, dash_capstyle="round")
    if add_cbar:
        fig.colorbar(
            im,
            ax=axs,
            orientation="vertical",
            extend="both",
            ticks=levels,
            pad=0.01,
        )

    fig.suptitle(
        "\n".join(
            filter(
                None,
                [
                    f"{diag.title} / {diag.units} ({PERIOD_STR})",
                    "& Horizontal Wind Streamlines" if overlay_stream else "",
                    f"Level: {unit_format(level * PA_TO_BAR, 'bar')}",
                ],
            )
        ),
        fontsize="medium",
    )

    if SAVE_FIGS:
        figsave(
            fig,
            paths.drafts / "xsec" / f"{GROUP_KEY}__{diag_key}_ll_{level:.0f}pa"
            f"__days_{START_DAY}_{END_DAY}",
        )
    return fig

In [ ]:
for diag_key in ["u", "temp"]:
    plot_maps(diag_key)